# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is specified via its Croissant schema URL, which describes the structure, fields, and assets of the dataset.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore its description and structure using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Use as single object

# Print dataset summary
print(f"Dataset name: {getattr(metadata, 'name', None)}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All exploration and extraction below is based on these canonical identifiers.

In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets (by @id):")
record_sets = [rs for rs in getattr(metadata, 'recordSet', [])]
if not record_sets:
    print("Warning: No record sets directly listed in metadata. Attempting to infer from distributions.")
    # Try to find record sets from distribution if present
    distributions = getattr(metadata, 'distribution', [])
    for dist in distributions:
        print(f" - {getattr(dist, '@id', None)}: {getattr(dist, 'name', None)} (type: {getattr(dist, '@type', None)})")
else:
    for rs in record_sets:
        print(f" - {getattr(rs, '@id', None)}: {getattr(rs, 'name', None)}")

# To find detailed schema, often look at file objects in distribution
print("\nPreviewing records by record set (automatically searches all listed):")
# Try to enumerate all record sets present in dataset.records
found_record_set_ids = set()
for record_set in dataset.list_record_sets():
    print(f"@id: {record_set} → first 1 record:")
    for i, rec in enumerate(dataset.records(record_set=record_set)):
        pprint.pprint(rec)
        if i >= 0:  # Only first record
            break
    found_record_set_ids.add(record_set)
if not found_record_set_ids:
    print("No data-ready record sets discovered in this Croissant package.")

## 3. Data Extraction
Load data from one or more record sets (using their `@id`) into Pandas DataFrames. The example below loads *all* available record sets discovered in the overview above.

In [ ]:
# Extract records from each available record set
dataframes = {}
record_set_ids = list(dataset.list_record_sets())
if not record_set_ids:
    raise ValueError("No record sets with data found in this dataset.")
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head(2))
    else:
        print(f"No records found for record set {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply EDA steps such as filtering, normalization, and grouping. Select meaningful numeric and categorical fields by their `@id` (column name).

In [ ]:
# For demonstration, select the first available record set and relevant fields
import numpy as np

# Choose the first available DataFrame
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]
print(f"Using record set: {record_set_id}")

# Show columns with their names
print("\nAvailable columns:")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col}")

# Heuristically select a likely numeric field (e.g., 'mw' for molecular weight, 'num_peptides', etc.). Adjust as appropriate based on actual columns.
# The column names are the field @ids.
def guess_numeric_column(df):
    for col in df.columns:
        if any(s in col.lower() for s in ['mw', 'molecular', 'peptide', 'count', 'value', 'intensity', 'num', 'abundance']):
            try:
                # Try converting to numeric
                as_num = pd.to_numeric(df[col], errors='coerce')
                if as_num.notnull().sum() > 0:
                    return col
            except Exception:
                continue
    # fallback
    return df.columns[0]

numeric_field = guess_numeric_column(df)
print(f"Selected numeric field for analysis: {numeric_field}")

# Convert to numeric (coerce errors)
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Example threshold filter (e.g., MW > 30000 or peptide_count > threshold)
threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
filtered_df = df[df[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalize field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to pick a categorical grouping field
def guess_group_field(df):
    # Exclude numeric field
    for col in df.columns:
        if col != numeric_field and df[col].nunique() < 20:
            return col
    return None

group_field = guess_group_field(df)
if group_field:
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(grouped_df.head())

## 5. Visualization
Visualize distributions of a selected numeric field, or relationships between two fields (if present) using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=20, color="skyblue")
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If a group_field exists and is categorical, show boxplot
if group_field:
    plt.figure(figsize=(10,4))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we've loaded, explored, and performed basic EDA on mass spectrometry data of extracellular vesicles isolated from human mast cells using the `mlcroissant` library.

Key findings and steps:
- Loaded dataset structure and extracted records for all available record sets using their `@id`.
- Identified and filtered based on a relevant numeric field, applied normalization, and grouped data by a categorical field where possible.
- Visualized the distribution of numeric measurements and their variation across groups.

This workflow can be adapted to any Croissant-compatible dataset for efficient, reproducible FAIR data analysis.